In [107]:
!pip install pandas numpy 

Defaulting to user installation because normal site-packages is not writeable


In [108]:
import pandas as pd
import numpy as np

In [6]:
df = pd.read_csv("zepto_initial.csv",encoding="latin1")

In [7]:
df.head()

,Category,name,mrp,discountPercent,availableQuantity,discountedSellingPrice,weightInGms,outOfStock,quantity
0,Fruits & Vegetables,Onion,2500,16,3,2100,1000,False,1
1,Fruits & Vegetables,Tomato Hybrid,4200,16,3,3500,1000,False,1
2,Fruits & Vegetables,Tender Coconut,5100,15,3,4300,58,False,1
3,Fruits & Vegetables,Coriander Leaves,2000,15,3,1700,100,False,100
4,Fruits & Vegetables,Ladies Finger,1400,14,3,1200,250,False,250


In [8]:
df['name']     = df['name'].str.strip()
df['Category'] = df['Category'].str.strip()

In [10]:
df.shape[0]
#rows

3732

In [11]:
df.shape[1]
#columns

9

In [14]:
df.columns

Index(['Category', 'name', 'mrp', 'discountPercent', 'availableQuantity',
       'discountedSellingPrice', 'weightInGms', 'outOfStock', 'quantity'],
      dtype='str')

In [16]:
df.describe(include = "all")

,Category,name,mrp,discountPercent,availableQuantity,discountedSellingPrice,weightInGms,outOfStock,quantity
count,3732,3732,3732.000000,3732.000000,3732.000000,3732.000000,3732.000000,3732,3732.000000
unique,14,1674,NaN,NaN,NaN,NaN,NaN,2,NaN
top,Cooking Essentials,Arden Eggs White,NaN,NaN,NaN,NaN,NaN,False,NaN
freq,514,10,NaN,NaN,NaN,NaN,NaN,3279,NaN
mean,NaN,NaN,15680.117899,7.617095,4.008574,14192.834941,387.843783,NaN,213.270900
std,NaN,NaN,16088.807618,9.211733,2.203511,13850.726265,678.096509,NaN,194.730976
min,NaN,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000
25%,NaN,NaN,6000.000000,0.000000,2.000000,5500.000000,100.000000,NaN,50.000000
50%,NaN,NaN,11000.000000,6.000000,5.000000,10400.000000,225.000000,NaN,186.000000
75%,NaN,NaN,20000.000000,10.000000,6.000000,18400.000000,450.000000,NaN,340.000000


In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3732 entries, 0 to 3731
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   Category                3732 non-null   str  
 1   name                    3732 non-null   str  
 2   mrp                     3732 non-null   int64
 3   discountPercent         3732 non-null   int64
 4   availableQuantity       3732 non-null   int64
 5   discountedSellingPrice  3732 non-null   int64
 6   weightInGms             3732 non-null   int64
 7   outOfStock              3732 non-null   bool 
 8   quantity                3732 non-null   int64
dtypes: bool(1), int64(6), str(2)
memory usage: 237.0 KB


In [19]:
df.isnull().sum()  #checking null values

Category                  0
name                      0
mrp                       0
discountPercent           0
availableQuantity         0
discountedSellingPrice    0
weightInGms               0
outOfStock                0
quantity                  0
dtype: int64

In [31]:
df.duplicated().sum()  #checking Duplicate rows

np.int64(0)

In [29]:
df = df.drop_duplicates().reset_index(drop=True) #removing duplicates

In [89]:
len(df)

3729

In [88]:
#1 Check if mrp is 0 or negative
mrp_zero = df[df['mrp'] <= 0]

In [50]:
#remove the row with mrp 0
df = df[df['mrp'] > 0].reset_index(drop=True)
len(df)

3729

In [42]:
#2 discountedSellingPrice should always be <= MRP
price_violation = df[df['discountedSellingPrice'] > df['mrp']]
len(price_violation)

0

In [48]:
#3 weightInGms = 0 makes price per gram impossible
zero_weight = df[df['weightInGms'] <= 0]

In [51]:
#4 out of stock=True but available quantity > 0 
inconsistent_stock = df[(df['outOfStock'] == True) & (df['availableQuantity'] > 0)]
len(inconsistent_stock)

0

In [85]:
#5 Convert MRP and prices from paise to rupees
df['mrp'] = df['mrp']/100
df['discountedSellingPrice'] = df['discountedSellingPrice'] /100
df.head()

,Category,name,mrp,discountPercent,availableQuantity,discountedSellingPrice,weightInGms,outOfStock,quantity
0,Fruits & Vegetables,Onion,25.0,16,3,21.0,1000,False,1
1,Fruits & Vegetables,Tomato Hybrid,42.0,16,3,35.0,1000,False,1
2,Fruits & Vegetables,Tender Coconut,51.0,15,3,43.0,58,False,1
3,Fruits & Vegetables,Coriander Leaves,20.0,15,3,17.0,100,False,100
4,Fruits & Vegetables,Ladies Finger,14.0,14,3,12.0,250,False,250


In [117]:
df['mrp'] = df['mrp'].round(2)
df['discountedSellingPrice'] = df['discountedSellingPrice'].round(2)

In [118]:
#Feature Engg (adding new columns)
#1.discountAmount (amount saved per product)
df['discountAmount'] = (df['mrp'] - df['discountedSellingPrice']).round(2)

#2. pricePer100g
df['pricePer100g'] = np.where(df['weightInGms'] > 0, 
                              (df['discountedSellingPrice']/df['weightInGms'] * 100).round(2),
                              np.nan)

#3 stockStatus
def stock_status(row):
    if row['outOfStock']:
        return 'Out of Stock'
    elif row['availableQuantity'] <= 2:
        return 'Low Stock'
    elif row['availableQuantity'] <= 4:
        return 'Medium Stock'
    else:
        return 'Well Stocked'

df['stock_status'] = df.apply(stock_status, axis=1)
df.head()

,Category,name,mrp,discountPercent,availableQuantity,discountedSellingPrice,weightInGms,outOfStock,quantity,discountAmount,pricePer100g,stock_status
0,Fruits & Vegetables,Onion,25.0,16,3,21.0,1000,False,1,4.0,2.10,Medium Stock
1,Fruits & Vegetables,Tomato Hybrid,42.0,16,3,35.0,1000,False,1,7.0,3.50,Medium Stock
2,Fruits & Vegetables,Tender Coconut,51.0,15,3,43.0,58,False,1,8.0,74.14,Medium Stock
3,Fruits & Vegetables,Coriander Leaves,20.0,15,3,17.0,100,False,100,3.0,17.00,Medium Stock
4,Fruits & Vegetables,Ladies Finger,14.0,14,3,12.0,250,False,250,2.0,4.80,Medium Stock


In [119]:
df.columns

Index(['Category', 'name', 'mrp', 'discountPercent', 'availableQuantity',
       'discountedSellingPrice', 'weightInGms', 'outOfStock', 'quantity',
       'discountAmount', 'pricePer100g', 'stock_status'],
      dtype='str')

In [120]:
!pip install psycopg2-binary sqlalchemy

Defaulting to user installation because normal site-packages is not writeable


In [121]:
from sqlalchemy import create_engine

In [122]:
username="postgres"
password="9568"
host= "localhost"
port = "5432"
database = "zepto"

engine = create_engine(f'postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}')

In [123]:
table_name="retail"
df.to_sql(table_name,engine,if_exists="replace",index=False)
print(f"data successfully loaded into table '{table_name}' in database '{database}'.")

data successfully loaded into table 'retail' in database 'zepto'.


In [124]:
df.to_csv("zepto_final.csv")